# Compact GRU training

Train four registered CUDA GRU settings across seeds 17, 29 and 43 through shared Python modules. Scaling uses training only; validation selects the setting by mean seed MAE. Seed 17 remains the primary artifact. Calibration and final test remain locked. No model code lives in this notebook.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import sys
import pandas as pd

here = Path.cwd().resolve()
repository = next(path for path in [here, *here.parents] if (path / 'docs/data/prepared-release.json').is_file())
if str(repository) not in sys.path:
    sys.path.insert(0, str(repository))
from ml.training.gru import run_gru

In [ ]:
run_id = datetime.now(timezone.utc).strftime('gru-notebook-%Y%m%dT%H%M%S%fZ')
output = repository / 'ml/runs' / run_id
report = run_gru(
    repository / 'ml/training/configs/do_pilot.json',
    repository / 'ml/training/configs/gru_settings.json',
    output, repository=repository, matrix=True,
)

In [ ]:
pd.DataFrame([{'configuration': case['configuration'], 'horizon_hours': case['horizon_hours'], 'selected_setting': case['selected']['setting'] if case['selected'] else None, 'mean_validation_mae_mgL': case['selected']['summary']['mean']['mae_mgL'] if case['selected'] else None, 'model_qualified': False} for case in report['cases']])
